# probit-регрессия: LR-тест, совместная значимость

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2 # критические значения chi2-распределения

# Не показывать FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# импорт датасета
df = pd.read_csv('../../datasets/TableF5-1.csv')
df

## Спецификация и подгонка модели
Для датасета `mroz_Greene` рассморим probit-регрессию **LFP на WA, I(WA ** 2), WE, KL6, K618, CIT, UN, log(FAMINC)**.

In [ ]:
mod = smf.probit(formula='LFP ~ 1 + WA + I(WA ** 2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data=df) # спецификация модели
res = mod.fit() # подгонка модели
res.nobs # число наблюдений, на которых была подогнана модель

## LR-тест: совместная значимость
Тестируем значимость влияния возраста жены, т.е. $$H_0:\beta_{WA}=\beta_{WA^2}=0.$$

Тестовая статистика $LR=2(\log L-\log L_{r})$ где 
- $\log L$ – лог-правдоподобие для исходной регрессии
- $\log L_{r}$ - лог-правдоподобие для регрессии с ограничениями (без переменных из гипотезы)

Для вычисления тестовой статистики нужно оценить две вложенных модели: без ограничений и с ограничениями

## <span style="color:#59afe1">**Важно**: обе регрессия д.б. оценены на одном и том же датасете! </span>

## Уровень значимости
Пусть уровень значимости $\alpha=10\%=0.1$

In [ ]:
# подгонка модели с ограничениями
df_mod = df[['LFP', 'WA', 'WE', 'KL6', 'K618', 'CIT', 'UN', 'FAMINC']].dropna()
mod_r = smf.probit(formula ='LFP ~ 1 + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data = df_mod)
res_r = mod_r.fit()
res_r.nobs # число наблюдений, на которых была подогнана модель

### Тестовая статистика LR-теста 

In [ ]:
# Тестовая статистика LR-теста с округленим
lr_stat=2*(res.llf-res_r.llf)
lr_stat.round(3)

### Степени свободы для $\chi^2$-распределения 

In [ ]:
res.df_model-res_r.df_model

### P-значение тестовой статистики LR-теста

In [ ]:
# P-значение тестовой статистики LR-теста с округленим
lr_pvalue = chi2.sf(lr_stat, df=res.df_model-res_r.df_model)
lr_pvalue.round(3)

### Критическое значение $\chi_{df}^2(\alpha)$

In [ ]:
sign_level = 0.1 # уровень значимости
chi2.isf(q=sign_level, df=res.df_model-res_r.df_model).round(3) 

### Значимо ли возраста жены?

`влияние возраста жены значимо` $(LR>\chi_{df}^2(\alpha))$

In [ ]:
# Статистика теста Вальда и её P-значение
res.wald_test('WA=I(WA ** 2)=0')